# Nodule segmentation: 128³ crop vs. 64³ crop + 8 neighbours (3×3 tiles) with smoothing

For one selected nodule this notebook:

1. **Selects** a nodule from your table (NLST cohort CSV with `coordX/Y/Z`, or `val_paths.csv` with masks).
2. **128³ crop:** preprocesses the CT exactly like training (LPS, 1 mm isotropic, HU clipped to [-1000, 400]), crops 128×128×128 around the nodule and segments all 128 axial slices (128×128 each) with your trained 2D V-Net.
3. **64³ crop + 8 neighbours:** for every slice of the 64³ crop around the nodule, also segments the 8 surrounding 64×64 tiles (3×3 grid, 192×192 mm), stitches the 9 probability maps and **smooths them together** (HMM along z/y/x, or Gaussian), then post-processes.
4. **Visualizes** both results: orthogonal views, a tile/seam view, a slice browser and a summary table.

Post-processing is identical to `04_infer_test_cohort.py`: threshold → per-slice hole filling → 3D connected components → drop tiny / single-slice blobs → keep the component nearest the nodule centre.

Only the **Configuration** cell needs editing. `ipywidgets` is optional (dropdown + slice slider); without it the notebook falls back to plain variables and static figures.

## Configuration

In [ ]:
from pathlib import Path

# folder that contains common.py, vnet2d.py, infer_lib.py (your training / inference code)
CODE_DIR = "/radraid2/mvinet/Lung/Current/code/09_29_26_model_retraining"
CKPT     = f"{CODE_DIR}/run/best.pt"                     # trained 2D V-Net checkpoint

# nodule table: needs image_path + either coordinates (LPS mm) or a mask column
TABLE      = "/path/to/temporal_nodule_volume_analysis_09032026.csv"   # or f"{CODE_DIR}/run/val_paths.csv"
COORD_COLS = ("coordX", "coordY", "coordZ")   # physical LPS mm; if absent, the mask centroid is used
MASK_COL   = "mask_path"                      # optional ground truth (ignored if the column is absent)
GT_MODE    = "own"   # "own" = only this nodule's mask (matches KEEP="center"); "union" = all masks of the CT
ID_COLS    = ("pid", "study_yr", "nodule_id") # columns used to label nodules (whichever exist)

# post-processing (same defaults as step 4)
THR, MIN_VOXELS, MIN_SLICES = 0.5, 8, 2
KEEP = "center"            # "center" | "all" | "largest"

# step 2: single big crop
BIG = 128                  # 128 x 128 x 128

# step 3: 64^3 crop + its 8 in-plane neighbours
TILE        = 64           # tile size (= training input size)
TILE_STRIDE = 64           # 64 = 9 adjacent tiles (192 mm); < 64 = overlapping tiles, Gaussian-blended
TILE_DEPTH  = 64           # slices of the 64^3 crop
SMOOTHING   = "hmm"        # "hmm" | "gaussian" | "none"
HMM_STAY    = 0.95         # HMM self-transition prob. (closer to 1 = bridges longer gaps)
GAUSS_SIGMA = 1.0          # mm, for SMOOTHING = "gaussian"

BATCH   = 256              # slices per forward pass (lower if GPU/CPU memory is tight)
DEVICE  = None             # None = cuda if available, else cpu
VIEW_HALF_MM = 40          # zoom: half-width of the displayed window around the nodule (None = full canvas)
SAVE_DIR = None            # e.g. f"{CODE_DIR}/fov_comparison" to save figures + NIfTIs; None = don't save

## Setup

In [ ]:
import sys, time
import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import ndimage

sys.path.insert(0, CODE_DIR)
from common import (HU_MIN, centroid_zyx, crop_start, crop_with_pad, crop_to_sitk,
                    load_ct_1mm, normalize_hu, read_path_table, resample_mask_to_ref)
from infer_lib import load_model, predict_volume, postprocess, size_measurements, grit_level

try:
    import ipywidgets as W
    from IPython.display import display
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False

device = torch.device(DEVICE or ("cuda" if torch.cuda.is_available() else "cpu"))
model = load_model(CKPT, device)
print(f"model loaded from {CKPT} on {device} | widgets: {HAVE_WIDGETS} | "
      f"{'GRIT 3-channel model' if model.grit is not None else '1-channel model'}")

# colours: GT blue, 128 crop orange (solid), 3x3 tiles magenta (dashed) - line style is a 2nd cue
C_GT, C_128, C_TILE = "#35b8ff", "#ff7a1a", "#e040a0"

## 1. Select a nodule

In [ ]:
df = read_path_table(TABLE).reset_index(drop=True)
has_coords = all(c in df.columns for c in COORD_COLS)
has_mask = MASK_COL in df.columns
if not has_coords and not has_mask:
    raise ValueError(f"{TABLE} needs coordinate columns {COORD_COLS} or a '{MASK_COL}' column")

id_cols = [c for c in ID_COLS if c in df.columns]
def label(i, r):
    tag = " | ".join(f"{c}={r[c]}" for c in id_cols) if id_cols else Path(r["image_path"]).parent.name
    return f"[{i}] {tag}"
labels = [label(i, r) for i, r in df.iterrows()]
print(f"{len(df)} nodules in {TABLE}  (centre from {'coordinates' if has_coords else 'mask centroid'})")

SELECT = None   # set to a row index (int) to skip the dropdown, e.g. SELECT = 12
picker = None
if HAVE_WIDGETS and SELECT is None:
    picker = W.Dropdown(options=list(zip(labels, df.index)), description="nodule",
                        layout=W.Layout(width="70%"))
    display(picker)
    print("pick a nodule, then run the next cells")
else:
    display(df.head(20)) if "display" in dir() else print(df.head(20))

In [ ]:
row_idx = SELECT if SELECT is not None else (picker.value if picker is not None else 0)
row = df.loc[row_idx]
print("selected:", labels[row_idx])
print("image:", row["image_path"])

# ---- preprocessing: identical to training (LPS, 1 mm iso, HU clip)
t0 = time.time()
ref, hu = load_ct_1mm(row["image_path"])
print(f"CT resampled to 1 mm: {hu.shape} (z, y, x)  in {time.time() - t0:.1f}s")

# ground truth: union of all masks of this CT (as in training), if available
gt_full = None
if has_mask:
    if GT_MODE == "own":
        paths = [str(row[MASK_COL])]
    else:
        paths = list(df.loc[df.image_path == row["image_path"], MASK_COL].astype(str).unique())
    paths = [p for p in paths if p and p.lower() != "nan" and Path(p).exists()]
    if paths:
        gt_full = np.zeros(hu.shape, np.uint8)
        for p in paths:
            gt_full |= resample_mask_to_ref(p, ref)
        print(f"ground truth: {GT_MODE} ({len(paths)} mask{'s' if len(paths) > 1 else ''})")

# nodule centre on the 1 mm grid, (z, y, x)
if has_coords and pd.notna(row[COORD_COLS[0]]):
    idx = ref.TransformPhysicalPointToContinuousIndex([float(row[c]) for c in COORD_COLS])
    center = np.array([idx[2], idx[1], idx[0]])
    src = "coordinates (LPS)"
else:
    own = resample_mask_to_ref(row[MASK_COL], ref)
    center = centroid_zyx(own)
    src = "mask centroid"
inside = bool(np.all(center >= 0) and np.all(center < np.array(hu.shape)))
print(f"centre (z, y, x) = {np.round(center, 1)} from {src}; inside CT: {inside}")

## 2. Single 128 × 128 × 128 crop
The model is fully convolutional and works slice by slice, so it runs directly on 128 × 128 slices (it was trained on 64 × 64).

In [ ]:
def segment(img_hu, center_in_crop):
    """HU crop (D, H, W) -> (probability, post-processed mask)."""
    pct = grit_level(model, img_hu, center_in_crop)          # None for 1-channel models
    prob = predict_volume(model, normalize_hu(img_hu), device, batch=BATCH, grit_pct=pct)
    pred = postprocess(prob, THR, MIN_VOXELS, MIN_SLICES, KEEP, fill_holes=True, center=center_in_crop)
    return prob, pred

size_big = (BIG, BIG, BIG)
start_big = np.array(crop_start(center, size_big))
img_big = crop_with_pad(hu, start_big, size_big, HU_MIN).astype(np.float32)
t0 = time.time()
prob_big, pred_big = segment(img_big, center - start_big)
print(f"128^3 crop: {pred_big.sum()} voxels predicted   ({time.time() - t0:.1f}s)")

## 3. 64³ crop + its 8 neighbours, smoothed together
For each of the 64 slices, the 64 × 64 crop and the 8 surrounding 64 × 64 tiles (3 × 3 grid) are segmented separately — every tile is exactly the training input size. The 9 probability maps are stitched (overlaps, if `TILE_STRIDE < 64`, are blended with Gaussian weights) and then smoothed:

* **`hmm`** – a two-state (background / nodule) hidden Markov model is run along every z, y and x line (forward–backward, network probability as the emission), and the three posteriors are averaged. Short dips in probability between confident nodule regions (tile seams, a weak slice) are bridged; isolated specks are suppressed. `HMM_STAY` sets how long a gap can be bridged.
* **`gaussian`** – a 3D Gaussian blur of the stitched probabilities (`GAUSS_SIGMA` mm).
* **`none`** – stitching only.

The plain 64³ result (centre tile only, no neighbours) is kept as a reference.

In [ ]:
def tile_offsets(stride, n=3):
    return [i * stride for i in range(n)]

def split_tiles(region, tile, stride):
    offs = tile_offsets(stride)
    return [region[:, y:y + tile, x:x + tile] for y in offs for x in offs]       # 9 x (D, tile, tile)

def blend_window(tile, stride):
    if stride >= tile:
        return np.ones((tile, tile), np.float32)
    g = np.exp(-0.5 * ((np.arange(tile) - (tile - 1) / 2) / (tile / 4)) ** 2)
    return np.maximum(np.outer(g, g), 1e-3).astype(np.float32)

def stitch(tile_probs, tile, stride):
    D = tile_probs[0].shape[0]
    R = 2 * stride + tile
    acc, wsum = np.zeros((D, R, R), np.float32), np.zeros((R, R), np.float32)
    w = blend_window(tile, stride)
    k = 0
    for y in tile_offsets(stride):
        for x in tile_offsets(stride):
            acc[:, y:y + tile, x:x + tile] += tile_probs[k] * w
            wsum[y:y + tile, x:x + tile] += w
            k += 1
    return acc / np.maximum(wsum, 1e-6)

def hmm_posterior(p, stay, axis):
    """2-state HMM forward-backward along `axis` for all lines at once -> P(nodule | line)."""
    x = np.moveaxis(np.clip(p, 1e-4, 1 - 1e-4), axis, 0).astype(np.float64)
    e = np.stack([1 - x, x], -1)
    T = np.array([[stay, 1 - stay], [1 - stay, stay]])
    L = x.shape[0]
    alpha = np.empty_like(e)
    a = 0.5 * e[0]; alpha[0] = a / a.sum(-1, keepdims=True)
    for t in range(1, L):
        a = (alpha[t - 1] @ T) * e[t]; alpha[t] = a / a.sum(-1, keepdims=True)
    post = np.empty(x.shape, np.float32)
    beta = np.ones_like(e[-1])
    g = alpha[-1] * beta; post[-1] = g[..., 1] / g.sum(-1)
    for t in range(L - 2, -1, -1):
        beta = (e[t + 1] * beta) @ T.T; beta /= beta.sum(-1, keepdims=True)
        g = alpha[t] * beta; post[t] = g[..., 1] / g.sum(-1)
    return np.moveaxis(post, 0, axis)

def smooth(prob):
    if SMOOTHING == "hmm":
        return np.mean([hmm_posterior(prob, HMM_STAY, a) for a in (0, 1, 2)], axis=0).astype(np.float32)
    if SMOOTHING == "gaussian":
        return ndimage.gaussian_filter(prob, GAUSS_SIGMA).astype(np.float32)
    return prob

R = 2 * TILE_STRIDE + TILE
size_reg = (TILE_DEPTH, R, R)
start_reg = np.array(crop_start(center, size_reg))
img_reg = crop_with_pad(hu, start_reg, size_reg, HU_MIN).astype(np.float32)

t0 = time.time()
tiles = split_tiles(img_reg, TILE, TILE_STRIDE)
stack = np.concatenate(tiles, axis=0)                                   # all 9 x 64 slices in one go
grit_pct_tiles = grit_level(model, img_reg, center - start_reg)   # one level for the nodule
p_all = predict_volume(model, normalize_hu(stack), device, batch=BATCH, grit_pct=grit_pct_tiles)
if grit_pct_tiles is not None:
    print(f"GRIT threshold level for this nodule: top {grit_pct_tiles:g}% of each tile")
tile_probs = np.split(p_all, 9, axis=0)
prob_stitched = stitch(tile_probs, TILE, TILE_STRIDE)
prob_smooth = smooth(prob_stitched)
c_reg = center - start_reg
pred_tiles = postprocess(prob_smooth, THR, MIN_VOXELS, MIN_SLICES, KEEP, fill_holes=True, center=c_reg)
print(f"3x3 tiles ({R}x{R}, {TILE_DEPTH} slices, stride {TILE_STRIDE}, smoothing={SMOOTHING}): "
      f"{pred_tiles.sum()} voxels   ({time.time() - t0:.1f}s)")

# reference: plain 64^3 crop = centre tile only, no neighbours, no smoothing
o = TILE_STRIDE
start_64 = start_reg + np.array([0, o, o])
pred_64 = postprocess(tile_probs[4], THR, MIN_VOXELS, MIN_SLICES, KEEP, fill_holes=True,
                      center=center - start_64)
print(f"plain 64^3 crop (centre tile only): {pred_64.sum()} voxels")

## 4. Visualize

### Summary

In [ ]:
# put everything on one common 1 mm canvas covering both crops
lo = np.minimum(start_big, start_reg)
hi = np.maximum(start_big + np.array(size_big), start_reg + np.array(size_reg))
shape = tuple(hi - lo)

def to_canvas(arr, start):
    out = np.zeros(shape, arr.dtype)
    s = np.array(start) - lo
    out[s[0]:s[0] + arr.shape[0], s[1]:s[1] + arr.shape[1], s[2]:s[2] + arr.shape[2]] = arr
    return out

ct_c = crop_with_pad(hu, lo, shape, HU_MIN)
m_big = to_canvas(pred_big.astype(np.uint8), start_big)
m_til = to_canvas(pred_tiles.astype(np.uint8), start_reg)
m_64 = to_canvas(pred_64.astype(np.uint8), start_64)
gt_c = crop_with_pad(gt_full, lo, shape, 0) if gt_full is not None else None
c_c = center - lo

def dice(a, b):
    a, b = a > 0, b > 0
    return 2 * (a & b).sum() / max(a.sum() + b.sum(), 1)

def touches(mask, m=3):
    return bool(mask.any() and (mask[:m].any() or mask[-m:].any() or mask[:, :m].any() or mask[:, -m:].any()
                                or mask[:, :, :m].any() or mask[:, :, -m:].any()))

rows = []
for name, m, local in [("plain 64^3 crop", m_64, pred_64), ("128^3 crop", m_big, pred_big),
                       (f"3x3 tiles + {SMOOTHING}", m_til, pred_tiles)]:
    r = dict(method=name, **size_measurements(m), touches_crop_border=touches(local))
    if gt_c is not None:
        r["dice_vs_gt"] = dice(m, gt_c)
    rows.append(r)
summary = pd.DataFrame(rows).set_index("method").round(2)
display(summary) if "display" in dir() else print(summary)
print(f"Dice(128^3, 3x3 tiles) = {dice(m_big, m_til):.3f}")

### Orthogonal views through the nodule centre

In [ ]:
VIEW_AT = "pred"       # "pred" = centroid of the predicted nodule (both methods), "center" = given centre
both = (m_big > 0) | (m_til > 0)
vc = c_c if (VIEW_AT == "center" or not both.any()) else np.array(ndimage.center_of_mass(both))
vc = np.clip(np.round(vc).astype(int), 0, np.array(shape) - 1)

# 2D window per view: (row axis, col axis) of the canvas; rows/cols kept within +-VIEW_HALF_MM of vc
AXES = {"axial": (1, 2), "coronal": (0, 2), "sagittal": (0, 1)}
def window(view):
    out = []
    for ax in AXES[view]:
        if VIEW_HALF_MM is None:
            out.append((0, shape[ax]))
        else:
            a = int(np.clip(vc[ax] - VIEW_HALF_MM, 0, max(shape[ax] - 2 * VIEW_HALF_MM, 0)))
            out.append((a, min(a + 2 * VIEW_HALF_MM, shape[ax])))
    return out

def planes(vol, c):
    full = {"axial": vol[c[0]], "coronal": vol[:, c[1], :], "sagittal": vol[:, :, c[2]]}
    out = {}
    for v, img in full.items():
        (r0, r1), (q0, q1) = window(v)
        out[v] = img[r0:r1, q0:q1]
    return out

def outline(ax, m, color, ls="-", lw=1.6):
    if m.any():
        ax.contour(m.astype(float), levels=[0.5], colors=color, linewidths=lw, linestyles=ls)

def tile_grid(ax, view):
    """Draw the 3x3 tile boundaries (in-plane) and the 64-slice extent (z) on the (zoomed) views."""
    (r0, _), (q0, _) = window(view)
    s = (start_reg - lo).astype(int)          # region start on the canvas, (z, y, x)
    ra, ca = AXES[view]                       # canvas axes shown as rows / columns
    s[ra] -= r0                               # shift into the zoomed window
    s[ca] -= q0
    ys = [s[1] + k * TILE_STRIDE for k in range(3)] + [s[1] + 2 * TILE_STRIDE + TILE]
    xs = [s[2] + k * TILE_STRIDE for k in range(3)] + [s[2] + 2 * TILE_STRIDE + TILE]
    kw = dict(color="white", lw=0.6, alpha=0.5, ls=":")
    xl, yl = ax.get_xlim(), ax.get_ylim()     # keep the zoom: lines outside the window must not widen it
    if view == "axial":
        for y in ys: ax.axhline(y - 0.5, **kw)
        for x in xs: ax.axvline(x - 0.5, **kw)
    else:
        for z in (s[0], s[0] + TILE_DEPTH): ax.axhline(z - 0.5, **kw)
        for v in (xs if view == "coronal" else ys): ax.axvline(v - 0.5, **kw)
    ax.set_xlim(xl); ax.set_ylim(yl)

def show(ax, img2d, view):
    ax.imshow(img2d, cmap="gray", vmin=-1000, vmax=400, interpolation="nearest",
              origin="lower" if view != "axial" else "upper")   # superior up in coronal/sagittal
    ax.set_xticks([]); ax.set_yticks([])

ctp = planes(ct_c, vc)
cols = ["axial", "coronal", "sagittal"]
fig, axes = plt.subplots(3, 3, figsize=(13, 13))
for j, v in enumerate(cols):
    for i, (title, m, col, ls) in enumerate([("128³ crop", m_big, C_128, "-"),
                                             (f"3×3 tiles + {SMOOTHING}", m_til, C_TILE, "--"),
                                             ("both", None, None, None)]):
        ax = axes[i, j]
        show(ax, ctp[v], v)
        if gt_c is not None:
            outline(ax, planes(gt_c, vc)[v], C_GT, lw=1.2)
        if m is not None:
            outline(ax, planes(m, vc)[v], col, ls)
        else:
            outline(ax, planes(m_big, vc)[v], C_128, "-")
            outline(ax, planes(m_til, vc)[v], C_TILE, "--")
        if i == 1:
            tile_grid(ax, v)
        ax.set_title(f"{title} — {v}", fontsize=10)
handles = [Line2D([], [], color=C_128, lw=2, label="128³ crop"),
           Line2D([], [], color=C_TILE, lw=2, ls="--", label=f"3×3 tiles + {SMOOTHING}")]
if gt_c is not None:
    handles.append(Line2D([], [], color=C_GT, lw=2, label="ground truth"))
fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False)
zoom = "full canvas" if VIEW_HALF_MM is None else f"±{VIEW_HALF_MM} mm"
fig.suptitle(f"{labels[row_idx]}   (views through the {'predicted nodule' if VIEW_AT == 'pred' else 'given centre'}, "
             f"{zoom}; dotted = tile boundaries)", x=0.01, ha="left")
fig.tight_layout(rect=(0, 0.03, 1, 0.97))
plt.show()

### Tiles, stitching and smoothing on one slice
Left to right: the 9 raw tile probabilities stitched side by side, the stitched map after `SMOOTHING`, and the final masks. Seams show up as discontinuities at the dotted lines.

In [ ]:
z_reg = int(np.clip(round(c_reg[0]), 0, TILE_DEPTH - 1))
raw_side = np.zeros((R, R), np.float32)                      # tiles placed without blending, for viewing
for k, (y, x) in enumerate([(y, x) for y in tile_offsets(TILE_STRIDE) for x in tile_offsets(TILE_STRIDE)]):
    raw_side[y:y + TILE, x:x + TILE] = tile_probs[k][z_reg]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))
for ax, (img, title) in zip(axes, [(img_reg[z_reg], "CT (192 mm region)"),
                                   (raw_side, "9 tile probabilities (raw)"),
                                   (prob_smooth[z_reg], f"stitched + {SMOOTHING}"),
                                   (img_reg[z_reg], "final masks")]):
    if title.startswith("CT") or title == "final masks":
        ax.imshow(img, cmap="gray", vmin=-1000, vmax=400, interpolation="nearest")
    else:
        h = ax.imshow(img, cmap="magma", vmin=0, vmax=1, interpolation="nearest")
        fig.colorbar(h, ax=ax, fraction=0.046, pad=0.02)
    for k in range(1, 3):
        ax.axhline(k * TILE_STRIDE - 0.5, color="white", lw=0.6, ls=":", alpha=0.6)
        ax.axvline(k * TILE_STRIDE - 0.5, color="white", lw=0.6, ls=":", alpha=0.6)
        ax.axhline(k * TILE_STRIDE + (TILE - TILE_STRIDE) - 0.5, color="white", lw=0.6, ls=":", alpha=0.6) if TILE_STRIDE < TILE else None
    ax.set_title(title, fontsize=10); ax.set_xticks([]); ax.set_yticks([])
# final masks in region coordinates: tiles result + 128 result mapped onto the region
off = start_big - start_reg
big_in_reg = np.zeros(img_reg.shape, np.uint8)
zs, ys, xs = [slice(max(o, 0), min(o + n, N)) for o, n, N in zip(off, size_big, img_reg.shape)]
big_in_reg[zs, ys, xs] = pred_big[zs.start - off[0]:zs.stop - off[0], ys.start - off[1]:ys.stop - off[1],
                                  xs.start - off[2]:xs.stop - off[2]]
outline(axes[3], big_in_reg[z_reg], C_128, "-")
outline(axes[3], pred_tiles[z_reg], C_TILE, "--")
if gt_full is not None:
    outline(axes[3], crop_with_pad(gt_full, start_reg, size_reg, 0)[z_reg], C_GT, lw=1.2)
fig.suptitle(f"slice z = {z_reg} of the 64-slice crop (through the nodule centre)", x=0.01, ha="left")
fig.tight_layout(); plt.show()

### Slice browser

In [ ]:
zmin, zmax = 0, shape[0] - 1
def draw_slice(z):
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    for a, (m, col, ls, t) in zip(ax, [(m_big, C_128, "-", "128³ crop"),
                                       (m_til, C_TILE, "--", f"3×3 tiles + {SMOOTHING}")]):
        (r0, r1), (q0, q1) = window("axial")
        show(a, ct_c[z, r0:r1, q0:q1], "axial")
        if gt_c is not None:
            outline(a, gt_c[z, r0:r1, q0:q1], C_GT, lw=1.2)
        outline(a, m[z, r0:r1, q0:q1], col, ls)
        a.set_title(f"{t} — z={z}  ({int(m[z].sum())} px)", fontsize=10)
    tile_grid(ax[1], "axial")
    plt.tight_layout(); plt.show()

if HAVE_WIDGETS:
    W.interact(draw_slice, z=W.IntSlider(value=int(vc[0]), min=zmin, max=zmax, continuous_update=False))
else:  # static montage over the predicted z-range
    zs_any = np.where((m_big.any(axis=(1, 2))) | (m_til.any(axis=(1, 2))))[0]
    for z in (zs_any[:: max(1, len(zs_any) // 6)] if len(zs_any) else [int(vc[0])]):
        draw_slice(int(z))

### Save (optional)

In [ ]:
if SAVE_DIR:
    out = Path(SAVE_DIR) / f"row{row_idx}"
    out.mkdir(parents=True, exist_ok=True)
    stub = ref  # 1 mm reference grid -> real-world geometry, overlays the CT in ITK-SNAP
    for name, arr, st in [("ct_128", img_big.astype(np.int16), start_big),
                          ("pred_128", pred_big.astype(np.uint8), start_big),
                          ("prob_128", prob_big.astype(np.float32), start_big),
                          ("ct_tiles", img_reg.astype(np.int16), start_reg),
                          ("pred_tiles", pred_tiles.astype(np.uint8), start_reg),
                          ("prob_tiles_raw", prob_stitched.astype(np.float32), start_reg),
                          ("prob_tiles_smooth", prob_smooth.astype(np.float32), start_reg)]:
        sitk.WriteImage(crop_to_sitk(arr, stub, st), str(out / f"{name}.nii.gz"))
    summary.to_csv(out / "summary.csv")
    print("saved to", out)
else:
    print("SAVE_DIR is None - nothing saved")